## ML Zoomcamp 2026 — Homework 2

[Official homework](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/02-regression/homework.md) · [Pinned 2026 dataset](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/f890391cf579ddf20decf9c67766b4a27e0b8ae1/cohorts/2026/data/car_fuel_efficiency_2026.csv)

All six questions were worked in Jupyter and saved on the course form on 8 October 2026. The Q3–Q6 code cells and outputs come from the saved Codespace working notebook.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/f890391cf579ddf20decf9c67766b4a27e0b8ae1/cohorts/2026/data/car_fuel_efficiency_2026.csv"
DATA_FILE = Path("car_fuel_efficiency_2026.csv")
columns = [
    "engine_displacement", "horsepower", "vehicle_weight",
    "model_year", "fuel_efficiency_mpg",
]
df = pd.read_csv(DATA_FILE if DATA_FILE.exists() else DATA_URL)[columns]
df.shape

### EDA

The `fuel_efficiency_mpg` histogram is roughly centered around 30 mpg; it does not show a pronounced long right tail.

In [ ]:
sns.histplot(df.fuel_efficiency_mpg, bins=50)
plt.xlabel("Fuel efficiency (mpg)")
plt.show()

### Question 1 — Missing values

**Answer: `horsepower`.** It has 877 missing values; the other four selected columns have none.

In [ ]:
df.isna().sum()

### Question 2 — Median horsepower

**Answer: 254.0** (the option **254**). The median uses the nonmissing `horsepower` values.

In [ ]:
df["horsepower"].median()

### Lesson 2.4 — 60/20/20 split

The course split with seed 42 gives 6,000 training, 2,000 validation, and 2,000 test rows. Before handling missing values, `X_train` has four columns and shape `(6000, 4)`; `y_train` has shape `(6000,)`.

In [ ]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

X_train = df_train.drop(columns=["fuel_efficiency_mpg"])
y_train = df_train["fuel_efficiency_mpg"]

print("train / validation / test:", len(df_train), len(df_val), len(df_test))
print("X_train:", X_train.shape, X_train.columns.tolist())
print("y_train:", y_train.shape)

### Question 3 — Missing horsepower

**Answer: With mean.** Training mean: 254.46338337605272. Validation RMSE rounded to three decimals: zero 2.205; mean 2.202.


In [9]:
features = [
    "engine_displacement", "horsepower",
    "vehicle_weight", "model_year",
]
target = "fuel_efficiency_mpg"
train_mean = df_train["horsepower"].mean()
print("train_mean:", train_mean)

train_mean: 254.46338337605272


In [10]:
def train_linear_regression(X, y):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X)
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()

for label, fill_value in [("zero", 0), ("training mean", train_mean)]:
    X_train = df_train[features].fillna({"horsepower": fill_value}).to_numpy()
    X_val = df_val[features].fillna({"horsepower": fill_value}).to_numpy()

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    print(label, round(rmse(y_val, y_pred), 3))

zero 2.205
training mean 2.202


### Question 4 — Regularization

**Answer: r = 0.** Validation RMSE: 2.2053. Missing values were filled with zero for every r.


In [11]:
def train_linear_regression_reg(X, y, r):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

X_train_zero = df_train[features].fillna(0).to_numpy()
X_val_zero = df_val[features].fillna(0).to_numpy()

results = []
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train_zero, y_train, r)
    y_pred = w0 + X_val_zero.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    results.append((score, r))
    print(f"r={r:>6}: RMSE={score:.4f}")

print("best r (smallest r breaks a tie):", min(results)[1])

r=     0: RMSE=2.2053
r=  0.01: RMSE=2.2058
r=   0.1: RMSE=2.2241
r=     1: RMSE=2.3492
r=     5: RMSE=2.4094
r=    10: RMSE=2.4195
r=   100: RMSE=2.4292
best r (smallest r breaks a tie): 0


### Question 5 — Split seed stability

**Answer: 0.029.** The unrounded standard deviation across ten validation RMSE scores is 0.028781274078155564.


In [12]:
scores = []

for seed in range(10):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    train = df.iloc[idx[:n_train]]
    val = df.iloc[idx[n_train:n_train + n_val]]

    X_train_seed = train[features].fillna(0).to_numpy()
    y_train_seed = train[target].to_numpy()
    X_val_seed = val[features].fillna(0).to_numpy()
    y_val_seed = val[target].to_numpy()

    w0, w = train_linear_regression(X_train_seed, y_train_seed)
    score = rmse(y_val_seed, w0 + X_val_seed.dot(w))
    scores.append(score)
    print(f"seed={seed}: RMSE={score:.6f}")

std = np.std(scores)
print("standard deviation:", std)
print("rounded Q5 value:", round(std, 3))

seed=0: RMSE=2.239204
seed=1: RMSE=2.202113
seed=2: RMSE=2.163420
seed=3: RMSE=2.204359
seed=4: RMSE=2.201880
seed=5: RMSE=2.245785
seed=6: RMSE=2.269721
seed=7: RMSE=2.190795
seed=8: RMSE=2.216611
seed=9: RMSE=2.207037
standard deviation: 0.028781274078155564
rounded Q5 value: 0.029


### Question 6 — Test evaluation

**Answer: 2.236.** The seed 9 split has 8,000 train-plus-validation and 2,000 test rows. With r=0.001 the test RMSE is 2.235827747194895.


In [13]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train_9 = df.iloc[idx[:n_train]]
df_val_9 = df.iloc[idx[n_train:n_train + n_val]]
df_test_9 = df.iloc[idx[n_train + n_val:]]

df_train_val_9 = pd.concat([df_train_9, df_val_9])

X_train_val = df_train_val_9[features].fillna(0).to_numpy()
y_train_val = df_train_val_9[target].to_numpy()
X_test = df_test_9[features].fillna(0).to_numpy()
y_test = df_test_9[target].to_numpy()

w0, w = train_linear_regression_reg(X_train_val, y_train_val, r=0.001)
test_rmse = rmse(y_test, w0 + X_test.dot(w))

print("train+validation / test:", len(df_train_val_9), len(df_test_9))
print("test RMSE:", test_rmse)
print("rounded Q6 value:", round(test_rmse, 3))

train+validation / test: 8000 2000
test RMSE: 2.235827747194895
rounded Q6 value: 2.236
